In [ ]:
import pathlib
import warnings

import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")  # Ignore all warnings
warnings.simplefilter("ignore")  # Additional suppression method

from notebook_init_utils.notebook_init_utils import init_notebook

root_dir, in_notebook = init_notebook()

In [ ]:
profile_dict = {
    "organoid_fs": {
        "input_profile_path": pathlib.Path(
            root_dir, "4.linear_modeling/results/linear_modeling/organoid_norm.parquet"
        ).resolve(strict=True),
        "metadata_columns": [
            "patient",
            "object_id",
            "unit",
            "dose",
            "treatment",
            "Target",
            "Class",
            "image_set",
            "Well",
            "Therapeutic_Categories",
            "single_cell_count",
        ],
    },
    "single_cell_fs": {
        "input_profile_path": pathlib.Path(
            root_dir, "4.linear_modeling/results/linear_modeling/sc_norm.parquet"
        ).resolve(strict=True),
        "metadata_columns": [
            "patient",
            "object_id",
            "unit",
            "dose",
            "treatment",
            "Target",
            "Class",
            "image_set",
            "Well",
            "Therapeutic_Categories",
            "parent_organoid",
        ],
    },
}

## Filter significant features
pvalue threshold is set to 0.05 - statistically significant features    
rsquared threshold is set to 0.5 - the explained variance is at least 50% of the total variance    
rsquared adjusted threshold is set to positive values - the model performs better than the mean    


### Single Cell

In [ ]:
df = pd.read_parquet(
    profile_dict["single_cell_fs"]["input_profile_path"],
)
print(df.shape)

(370500, 18)


In [ ]:
pvalue_threshold_max = 0.05  # significance threshold for p-values
rsquared_threshold_min = 0.5  # 50% of variance explained by the model
rsquared_adj_threshold_min = 0  # the model performs better than the null model
coefficient_threshold_min = 1  # minimum effect size of 1

In [ ]:
# filter significant features
df_filtered = df[
    (df["pvalue"] < pvalue_threshold_max)
    & (df["rsquared"] > rsquared_threshold_min)
    & (df["rsquared_adj"] > rsquared_adj_threshold_min)
    & (df["coefficient"].abs() > coefficient_threshold_min)
].copy()
print(df_filtered.shape)
df_filtered.head()

(65, 18)


,term,patient,treatment,drug,therapeutic_category,feature,rsquared,rsquared_adj,fvalue,pvalue,coefficient,intercept,feature_original,Compartment,Channel,Feature_type,Measurement,pvalue_fdr
23348,treatment,NF0014_T1,Staurosporine_10nM,Staurosporine,Experimental,Cytoplasm_ER_Intensity_MinIntensityEdge,0.505273,0.5043,519.27798,3.638666e-06,1.292335,-0.248176,Cytoplasm_ER_Intensity_MinIntensityEdge,Cytoplasm,ER,Intensity,MinIntensityEdge,1.834676e-05
23350,treatment,NF0016_T1,Staurosporine_10nM,Staurosporine,Experimental,Cytoplasm_ER_Intensity_MinIntensityEdge,0.505273,0.5043,519.27798,3.638666e-06,1.292335,-0.248176,Cytoplasm_ER_Intensity_MinIntensityEdge,Cytoplasm,ER,Intensity,MinIntensityEdge,1.834676e-05
23351,treatment,NF0018_T6,Staurosporine_10nM,Staurosporine,Experimental,Cytoplasm_ER_Intensity_MinIntensityEdge,0.505273,0.5043,519.27798,4.137657e-05,-1.273834,-0.248176,Cytoplasm_ER_Intensity_MinIntensityEdge,Cytoplasm,ER,Intensity,MinIntensityEdge,1.738340e-04
23354,treatment,NF0035_T1,Staurosporine_10nM,Staurosporine,Experimental,Cytoplasm_ER_Intensity_MinIntensityEdge,0.505273,0.5043,519.27798,4.096327e-41,1.536188,-0.248176,Cytoplasm_ER_Intensity_MinIntensityEdge,Cytoplasm,ER,Intensity,MinIntensityEdge,3.346736e-39
23359,treatment,SARCO361_T1,Staurosporine_10nM,Staurosporine,Experimental,Cytoplasm_ER_Intensity_MinIntensityEdge,0.505273,0.5043,519.27798,3.520179e-19,-1.233738,-0.248176,Cytoplasm_ER_Intensity_MinIntensityEdge,Cytoplasm,ER,Intensity,MinIntensityEdge,7.201227e-18


In [ ]:
df_filtered["treatment"].unique()

<ArrowStringArray>
['Staurosporine_10nM',      'Onalespib_1uM',     'Everolimus_1uM',
   'Binimetinib_10uM',  'Mirdametinib_10uM',    'Selumetinib_1uM',
        'Digoxin_1uM',     'Copanlisib_1uM',   'Fimepinostat_1uM',
   'Mirdametinib_1uM',     'Linsitinib_1uM',      'Rapamycin_1uM',
    'Binimetinib_1uM',        'ARV-825_1uM',    'Trabectedin_1uM']
Length: 15, dtype: str

In [ ]:
df_filtered["patient"].unique()

<ArrowStringArray>
[  'NF0014_T1',   'NF0016_T1',   'NF0018_T6',   'NF0035_T1', 'SARCO361_T1',
 'SARCO219_T2',   'NF0055_T1',   'NF0014_T2',   'NF0030_T1',   'NF0037_T1']
Length: 10, dtype: str

In [ ]:
df_filtered["feature"].unique()

<ArrowStringArray>
['Cytoplasm_ER_Intensity_MinIntensityEdge',
    'Nuclei_ER_Intensity_MaxIntensityEdge',
          'Cell_ER_Intensity_MinIntensity',
     'Cytoplasm_ER_Intensity_MinIntensity',
        'Nuclei_ER_Intensity_MinIntensity',
      'Cell_ER_Intensity_MinIntensityEdge']
Length: 6, dtype: str

### Organoid

In [ ]:
df = pd.read_parquet(
    profile_dict["organoid_fs"]["input_profile_path"],
)
print(df.shape)

(123500, 18)


In [ ]:
pvalue_threshold_max = 0.05
rsquared_threshold_min = 0.4
rsquared_adj_threshold_min = 0
coefficient_threshold_min = 1

In [ ]:
# filter significant features
df_filtered = df[
    (df["pvalue"] < pvalue_threshold_max)
    & (df["rsquared"] > rsquared_threshold_min)
    & (df["rsquared_adj"] > rsquared_adj_threshold_min)
    & (df["coefficient"].abs() > coefficient_threshold_min)
].copy()
print(df_filtered.shape)
df_filtered.head()

(434, 18)


,term,patient,treatment,drug,therapeutic_category,feature,rsquared,rsquared_adj,fvalue,pvalue,coefficient,intercept,feature_original,Compartment,Channel,Feature_type,Measurement,pvalue_fdr
522,treatment,NF0016_T1,Trametinib_1uM,Trametinib,Kinase Inhibitor,Organoid_AGP_Intensity_IntegratedIntensityEdge,0.423239,0.418909,97.739251,7.258216e-05,2.791396,0.329168,Organoid_AGP_Intensity_IntegratedIntensityEdge,Organoid,AGP,Intensity,IntegratedIntensityEdge,8.910582e-04
534,patient,NF0018_T6,Trametinib_1uM,Trametinib,Kinase Inhibitor,Organoid_AGP_Intensity_IntegratedIntensityEdge,0.423239,0.418909,97.739251,2.787296e-09,1.919862,0.329168,Organoid_AGP_Intensity_IntegratedIntensityEdge,Organoid,AGP,Intensity,IntegratedIntensityEdge,3.956431e-08
1834,patient,NF0018_T6,Trametinib_1uM,Trametinib,Kinase Inhibitor,Organoid_DNA_Granularity_6,0.414448,0.410029,93.782384,3.056775e-34,6.294688,-0.049911,Organoid_DNA_Granularity_6,Organoid,DNA,Granularity,6,2.711655e-32
1842,patient,SARCO361_T1,Trametinib_1uM,Trametinib,Kinase Inhibitor,Organoid_DNA_Granularity_6,0.414448,0.410029,93.782384,1.272516e-40,6.028190,-0.049911,Organoid_DNA_Granularity_6,Organoid,DNA,Granularity,6,1.490784e-38
2004,treatment,NF0016_T1,Trametinib_1uM,Trametinib,Kinase Inhibitor,Organoid_DNA_Intensity_IntegratedIntensity,0.419808,0.415452,96.373631,1.593908e-04,3.527704,0.331176,Organoid_DNA_Intensity_IntegratedIntensity,Organoid,DNA,Intensity,IntegratedIntensity,1.710999e-03


In [ ]:
df_filtered["treatment"].unique()

<ArrowStringArray>
[    'Trametinib_1uM', 'Staurosporine_10nM',      'Onalespib_1uM',
     'Everolimus_1uM',       'Imatinib_1uM',      'Ketotifen_1uM',
   'Binimetinib_10uM',  'Mirdametinib_10uM',    'Selumetinib_1uM',
        'Digoxin_1uM',     'Copanlisib_1uM',   'Fimepinostat_1uM',
    'Trametinib_10uM',   'Selumetinib_10uM',      'Nilotinib_1uM',
   'Mirdametinib_1uM',     'Linsitinib_1uM',      'Rapamycin_1uM',
   'Sapanisertib_1uM',        'ARV-825_1uM',    'Trabectedin_1uM',
    'Vistusertib_1uM',   'Panobinostat_1uM']
Length: 23, dtype: str

In [ ]:
df_filtered["patient"].unique()

<ArrowStringArray>
[  'NF0016_T1',   'NF0018_T6', 'SARCO361_T1',   'NF0035_T1',   'NF0055_T1',
   'NF0014_T2',   'NF0030_T1',   'NF0037_T1',   'NF0014_T1',   'NF0021_T1',
 'SARCO219_T2',   'NF0040_T1']
Length: 12, dtype: str

In [ ]:
df_filtered["feature"].unique()

<ArrowStringArray>
[       'Organoid_AGP_Intensity_IntegratedIntensityEdge',
                            'Organoid_DNA_Granularity_6',
            'Organoid_DNA_Intensity_IntegratedIntensity',
        'Organoid_DNA_Intensity_IntegratedIntensityEdge',
                    'Organoid_ER_Intensity_MinIntensity',
               'Organoid_NoChannel_AreaSizeShape_Volume',
            'Organoid_AGP_Intensity_IntegratedIntensity',
             'Organoid_ER_Intensity_IntegratedIntensity',
         'Organoid_ER_Intensity_IntegratedIntensityEdge',
           'Organoid_Mito_Intensity_IntegratedIntensity',
       'Organoid_Mito_Intensity_IntegratedIntensityEdge',
  'Organoid_DNA-AGP_Colocalization_MandersCoeffCostesM1',
 'Organoid_DNA-Mito_Colocalization_MandersCoeffCostesM1']
Length: 13, dtype: str